In [16]:
import json
import os
import random
import re
import requests
import scrapy
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from bs4 import BeautifulSoup
from furl import furl
from itemadapter import ItemAdapter
from itemloaders.processors import TakeFirst, Identity, Compose, MapCompose, Join
from lxml import html
from lxml.etree import tostring
from parsel import Selector
from scrapy.exceptions import DropItem
from scrapy.linkextractors import LinkExtractor
from scrapy.loader import ItemLoader
from scrapy.spiders import CrawlSpider, Rule
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.options import Options 
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from urllib.parse import urljoin

# Plus your own Scrapy items and itemloaders

## Task 1

### Helper function 1

In [21]:
#extract article links from a page
def extract_article_links(soup, base_url):
    """
    finds all valid wiki article links on a page
    returns set of full URLs
    
    sets automatically remove duplicates, easier to find
    intersections later (which children link to other children)
    """
    
    links = set()  #using set instead of list to avoid duplicate links
    
    #loop through all <a> tags that have href
    for a_tag in soup.find_all("a", href=True): #soup.find_all("a", href=True) means: find all anchor tags that have href
        href = a_tag["href"] #href=True filters out <a> tags without links
        
        ### filter out links we don't want:
        
        # skip anchor links (like #cite_note-1) these just jump to sections on same page
        if href.startswith("#"):
            continue
        
        # skip external links (like https://deepikapadukone.com/) i only want internal articles
        if href.startswith("http://") or href.startswith("https://"):
            continue
        
        # skip special Wikipedia pages (File: Special: Category:)
        # these aren't actual articles about topics
        if "File:" in href or "Special:" in href or "Category:" in href:
            continue
            
        # make full URL from relative link
        # urljoin() combines base_url + href
        full_url = urljoin(base_url, href)
        links.add(full_url)
    
    return links

### Helper function 2

In [119]:
# fetch a page and return beautifulsoup
def get_page_soup(url):
    """
    fetches a page and returns beautiful object
    returns none if page can't be fetched
    """
    try: #try/except is good for error handling, dont want a rquest to fail due server being down or connection issues
        # timeout=15 prevents hanging if page is slow
        response = requests.get(url, timeout=15) #wait 15 sec for response
        response.raise_for_status()  # raises error if status code is 4xx or 5xx *client errors and server errors* source for this idea is stackoverflow
        return BeautifulSoup(response.text, "html.parser") #response.text is raw html, soup makes it readable
    except:
        # ff anything goes wrong (timeout, 404) we return None
        # skip the page 
        return None

### Helper function 3

In [55]:
#build the child to child link graph
def build_link_graph(root_url, child_urls, base_url, include_root=False):
    """
    this function a graph showing which children link to which children
    returns a dictionary {child_url: set of children it links to}
    
    Graph structure: nodes = children, edges = links between them
    """

    # make empty dictionary to store the graph
    # dictionary structure {child_url: set of children it links to}
    # eg {"Filmfare": {"Filmfare_Awards", "Bollywood"}, "Filmfare_Awards": {"Filmfare"}}
    # this represents: Filmfare links to Filmfare_Awards and Bollywood, Filmfare_Awards links back to Filmfare
    
    graph = {}  
    
    #visit each child page, need to visit each child's page to see what THEY link to
    
    for child in child_urls:
        soup = get_page_soup(child) #downloads the html and converts it to soup
        if soup is None:
            # if we can't fetch the page, skip it
            continue
        
        # get all links from this child's page
        links_on_page = extract_article_links(soup, base_url)
        
        # don't want all links on the page, only links to OTHER children
        # eg if root has children [A, B, C, D] and we're looking at child A's page
        # A's page might link to B, C, Z, Y, X
        # we only care about links to B and C (other children)
        # we ignore Z, Y, X (not children of root)
        
        valid_targets = child_urls.copy()  # start with all children
        #.copy() creates a new set so we don't modify the original child_urls

        # special case: for "outgoing" direction, also count links back to root
        # purpose of include_root parameter 
        # for incoming: we count how many children link TO each child (no need for root)
        # for outgoing: we count how many links each child has, INCLUDING links back to root
        # so, if child links back to root article, that counts as an outgoing link
        if include_root:
            #add root to the set of valid targets
            valid_targets.add(root_url)
        
        # intersection() gives us only links that are BOTH on the page AND in valid_targets
        # this is the "only consider links to other children" requirement
        graph[child] = links_on_page.intersection(valid_targets)
    
    return graph

### Helper function 4

In [69]:
# count incoming links (indegree)
def count_incoming_links(graph):
    """
    function counts how many other articles link to each article
    returns dictionary: {article_url: count}
    
    incoming links = in-degree in graph theory (from lectures)
    need to reverse the graph edges to count this
    """

    #create empty dictionary to store incoming link counts
    # structure: {article_url: number of articles that link to it}
    # eg: {"Filmfare_Awards": 5} means 5 other children link to Filmfare_Awards
    incoming_counts = {}  


    # reverse the graph to show incoming links
    # the graph shows: child -> who they link to (outgoing edges)
    # but we need: child -> who links to them (incoming edges)
    # 
    # example of what we have so far(graph):
    #   A -> {B, C}    (A links to B and C)
    #   B -> {C}       (B links to C)
    #   C -> {A}       (C links to A)
    # 
    # wat we need to calculate (incoming):
    #   A: 1 incoming (C links to it)
    #   B: 1 incoming (A links to it)
    #   C: 2 incoming (A and B link to it)
    # 
    # we reverse the direction by looping through edges
    
    # loop through each article (source) and the set of articles it links to (targets)
    # graph.items() gives us pairs like: ("Filmfare", {"Filmfare_Awards", "Bollywood"})
    # source = "Filmfare", targets = {"Filmfare_Awards", "Bollywood"}
    # loop through each source article and its targets
    for source, targets in graph.items():
        # for each article that source links to, increas its incoming count
        for target in targets:
            # .get(target, 0) returns 0 if target not in dictionary yet
            # this is cleaner than checking "if target in dict" every time
            incoming_counts[target] = incoming_counts.get(target, 0) + 1
    
    return incoming_counts

### Helper function 5

In [77]:
#count outgoing links (outdegree)
def count_outgoing_links(graph):
    """
    function counts how many other articles each article links to
    returns dictionary: {article_url: count}
    
    outgoing links = out-degree 
    easier than incoming because we already have this in our graph.
    """

    # dictionary stores outgoing link counts
    # structure: {article_url: number of other children it links to}
    # eg: {"Filmfare": 10} means Filmfare links to 10 other children

    outgoing_counts = {}
    # graph is already structured as article -> set of articles it links to
    # so  just need to count the size of each set

    #loop through each article and its set of links
    for article, links in graph.items(): # graph.items() gives us pairs like: ("Filmfare", {"Filmfare_Awards", "Bollywood"})
        outgoing_counts[article] = len(links)  # len() of set gives us the count

        # eg len({"Filmfare_Awards", "Bollywood"}) = 2
        
        #  len() works here because links is a set of urls this article points to
        # the size of that set = number of outgoing links
    
    return outgoing_counts

### Helper function 6

In [121]:
# extract information from article
def extract_article_info(soup):
    """
    extracts key information from a Wikipedia article.
    returns dict with: first_paragraph, sections, table_count, largest_image
    
    """

    # empty dictionary to store all the info we extract
    # gonna add keys: first_paragraph, sections, table_count, largest_image
    info = {}
    
    ### Extract first paragraph
    
    first_para = "" #need empty string
    
    #Loop through all <p> (paragraph) tags to find the first real paragraph
    # looping because wikipedia articles have:
    # empty paragraphs
    # very short paragraphs 
    # the actual introduction paragraph
    #so we skip the junk and find the real first paragraph

    
    for p in soup.find_all("p"):

        #extract the text content from the <p> tag
        # .get_text(strip=True) does two things
        # extracts all text (removes html tags)
        # strip=True removes leading/trailing whitespace
        # eg: <p>  Hello <b>world</b>  </p> -> "Hello world"
        text = p.get_text(strip=True)
        
        #skip very short paragraphs (these are usually not the main text)
        if text and len(text) > 50:
            # remove citation numbers like [1], [2] with regex
            # re.sub(pattern, replacement, string) replaces matches
            # \[\d+\] means [followed by digits followed by]
            text = re.sub(r'\[\d+\]', '', text)
            first_para = text #Save this as our first paragraph

            break  # stop after finding first good paragraph
    
    info["first_paragraph"] = first_para #store the first paragraph in our info dictionary
    
    #### Extract section names

    #  empty list to store section names
    # sections are like table of contents, History, Reception, See also,
    sections = []

    #f ind all section headings
    # sections in our Kiwix are in <h2> tags
    for h2 in soup.find_all("h2"): 
        section_text = h2.get_text(strip=True)
        
        # some sections have [edit] links, remove those, wiki adds edit buttons next to headings "History[edit]"
        section_text = re.sub(r'\[edit\]', '', section_text).strip()
        
        # only add non-empty , after removing [edit], some headings can be empty
        if section_text:
            sections.append(section_text)
    #store the list of sections

    info["sections"] = sections
    
    ### Count tables
    # count all <table> tags in the article
    info["table_count"] = len(soup.find_all("table")) #if article has 3 tables, len(soup.find_all("table")) = 3
    
    ### Largest image
    largest_image = None # will store the caption/filename of largest image
    max_area = 0  # track the largest area we've seen (in pixels)
    
    # loop through all images to compare
    for img in soup.find_all("img"):
        
        # get width and height attributes
        # int() converts string to number, default to 0 if missing  
        width = int(img.get("width", 0))
        height = int(img.get("height", 0))
        area = width * height  # area in pixels
        
        # compare this image's area to the max we saw
        if area > max_area:
            max_area = area #update max_area so we compare future images against this on
            caption = None 
            
            # get caption text first 
            # images are sometimes inside <figure> tags with <figcaption>

            # find the parent <figure> element (if it exists)
            # .find_parent("figure") walks up html tree
            # returns the <figure> element, or None if img is not inside a figure
            parent = img.find_parent("figure")
            if parent:
                # found a parent <figure>, now look for <figcaption> inside it
                figcaption = parent.find("figcaption")
                if figcaption:
                    caption = figcaption.get_text(strip=True) #extract text when found
            
            # if no caption, use filename as fallback, not all images have captions, but all have filenames
            if not caption:
                src = img.get("src", "")
                # get src attribute (the image URL/path

                # extract just the filename from the path
                # .split("/") splits by / into a list of parts
                filename = src.split("/")[-1] #end of path gives jpg name
                #remove extension "image.jpg" -> "image"
                caption = filename.split(".")[0]
            #save this image's caption/filename as the larges
            largest_image = caption
    
    info["largest_image"] = largest_image
    
    return info

### Main function

In [120]:
def find_the_nexus(article_url, direction="incoming"):
    """
    finds the 'nexus' article among children of the root article.
    
    The nexus is the child with the most incoming or outgoing links
    depending on the direction parameter
    """
    
    #set base_url to the article URL itself
    # urljoin() needs a base URL to convert relative links to absolute URLs
    # eg: If article_url = "http://.../Deepika_Padukone"
    # and we find a link "Filmfare_Awards"
    # urljoin will combine them: "http://.../Filmfare_Awards"
    
    base_url = article_url
    

    #get root page 
    print(f"Analyzing: {article_url}")
    root_soup = get_page_soup(article_url)
    
    #error handling: if we can't get the root page, we can't continue
    if root_soup is None:
        print("error: could not get root page")
        return
    
    # extract all article links from the root page
    # these are the "children", the first level of our graph
    # eg: Deepika_Padukone page links to 126 different articles
    # extract_article_links() filters out junk (anchors, external links etc)
    # and returns only valid wikipedia article links 
    children = extract_article_links(root_soup, base_url)
    print(f"Found {len(children)} child articles")
    
    ### build graph of child-to-child links 
    #for outgoing direction, we include links back to root in the count
    
    include_root = (direction == "outgoing")

    # build the graph using our helper function
    # graph will be: {child_url: set of children it links to}
    # this visits each child's page and finds which other children they link to
    graph = build_link_graph(article_url, children, base_url, include_root)
    
    ### count links based on direction
    if direction == "incoming":
        #count how many children link TO each child (in-degree)
        #this reverses the graph edges to count incoming
        counts = count_incoming_links(graph)
    else:  # outgoig
        counts = count_outgoing_links(graph)
    # now counts is: {child_url: link_count}
    # eg: {"http://.../Filmfare": 70, "http://.../Bollywood": 61, ...}

    
    ### Sort and get top 10
    # sorted() with key=lambda sorts by the count (second element of tuple)
    # reverse=True gives us highest counts first
    sorted_articles = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    
    # take only top 10 for display
    top_10 = sorted_articles[:10]
    
    ###print top 10 results
    #extract article name from URL for cleaner display
    root_title = article_url.split("/")[-1].replace("_", " ") # .split("/")[-1] gets the last part after splitting by /
     # .replace("_", " ") replaces underscores with spaces

    print(f"\nFinding the Nexus of {root_title} for {direction} links:")


    # print each article in the top 10 with numbering
    # enumerate(top_10, 1) gives us: (1, first_item), (2, second_item)
    # starting from 1 (not 0) for readibility
    # each item in top_10 is a tuple (url, count)
    # so we unpack it
    
    for i, (url, count) in enumerate(top_10, 1):
        title = url.split("/")[-1].replace("_", " ") #same as above
        link_word = "link" if count == 1 else "links" #handle singular vs plural for grammatical correctness
        print(f"{i}. {title}: {count} {direction} {link_word}") #eg output: "1. Filmfare: 70 incoming links"
    
    ###  identify the nexus 
    
    #if all children had 0 links, top_10 might be empty
    if top_10:
        # get the first article from top_10
        # top_10[0] is the first tuple (url, count)
        # top_10[0][0] is the url part of that first tuple
        nexus_url = top_10[0][0]
        nexus_title = nexus_url.split("/")[-1].replace("_", " ")
        
        print(f"The Nexus of {root_title} for {direction} links is {nexus_title}.")
        
        #display detailed info about the nexus
        nexus_soup = get_page_soup(nexus_url)
        if nexus_soup:
            info = extract_article_info(nexus_soup) # info will be a dictionary with first_paragraph, sections, table_count, largest_image
            
            print(info["first_paragraph"])
            
            print()  #blank line for formatting
            
            #print sections
            if info["sections"]:
                sections_text = ", ".join(info["sections"])
                print(f"The Wikipedia article of {nexus_title} contains the sections {sections_text}.")
            else:
                print(f"The Wikipedia article of {nexus_title} contains no major sections.")
            
            #table count
            print(f"It contains a total of {info['table_count']} tables.")
            
            # largest image
            if info["largest_image"]:
                print(f'The largest image on the page is "{info["largest_image"]}".')

In [ ]:
root_url = "http://192.168.1.151:9123/wikipedia_en_indian-cinema_maxi_2025-10/Deepika_Padukone"

# directions: "incoming" and "outgoing"
find_the_nexus(root_url, direction="outgoing")